#  Resource Allocation Model — Sri Lanka District Poverty
### Generates risk percentages per district and distributes budget equitably

**Pipeline Overview**
1. Install & Import dependencies
2. Load & preprocess `Povertylines.csv`
3. Feature Engineering & Normalisation
4. Enhanced Risk Score with Custom Rules
5. Budget Allocation Engine
6. Allocation Report & Visualisation
7. Interactive Budget Input

> Upload `Povertylines.csv` via the Colab Files panel before running.

##  Section 1 — Install & Import Dependencies

In [1]:
!pip install sentence-transformers scikit-learn matplotlib seaborn pandas numpy --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from sentence_transformers import SentenceTransformer
import pickle
warnings.filterwarnings("ignore")
from sklearn.preprocessing import MinMaxScaler
from matplotlib.patches import Patch

sns.set_theme(style="whitegrid", palette="muted")
print(" Libraries loaded.")

 Libraries loaded.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

##  Section 2 — Load & Preprocess Data

Loads `Povertylines.csv`, standardises column names, strips comma-formatted numbers,
and fixes known anomalies (Kilinochchi Gini = 0, Gampaha per-capita income = 269).

In [4]:
FILE_PATH = "/content/drive/MyDrive/DSGP/poverty/Povertylines.xlsx"
df_raw = pd.read_excel(FILE_PATH)

df = df_raw.copy()
df.columns = (df.columns.str.strip().str.lower()
               .str.replace(r"\s+", "_", regex=True)
               .str.replace(r"[().]", "", regex=True)
               .str.replace(r"_+", "_", regex=True))

rename_map = {
    "mean_household_income_per_month":                 "mean_hh_income",
    "median_household_income_per_month_rs":            "median_hh_income",
    "average_household_size":                          "avg_hh_size",
    "gini_coefficient_income":                         "gini_income", # Corrected key
    "mean_per_capita_income_per_month_rs":             "mean_per_capita_income",
    "mean_household_expenditure_per_month_rs":         "mean_hh_expenditure",
    "median_household_expenditure_per_month_rs":       "median_hh_expenditure",
    "gini_coefficient_expenditure":                    "gini_expenditure", # Corrected key
    "mean_household_per_capita_expenditure_per_month": "mean_hh_per_capita_expenditure",
}
df.rename(columns=rename_map, inplace=True)

for col in [c for c in df.columns if c != "district"]:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce")

# Fix anomalies
median_gini = df.loc[df["gini_income"] > 0, "gini_income"].median()
df.loc[df["gini_income"] == 0, "gini_income"] = median_gini
median_gini_exp = df.loc[df["gini_expenditure"] > 0, "gini_expenditure"].median()
df.loc[df["gini_expenditure"] == 0, "gini_expenditure"] = median_gini_exp
mask = df["district"] == "Gampaha"
df.loc[mask, "mean_per_capita_income"] = (
    df.loc[mask, "mean_hh_income"] / df.loc[mask, "avg_hh_size"]).round(0)

price_cols = [c for c in df.columns if c.startswith("202")]
hh_cols = ["mean_hh_income","median_hh_income","avg_hh_size","gini_income",
           "mean_per_capita_income","mean_hh_expenditure","median_hh_expenditure",
           "gini_expenditure","mean_hh_per_capita_expenditure"]

print(f" Loaded & cleaned: {df.shape[0]} districts × {df.shape[1]} columns")
df[["district"] + hh_cols]


 Loaded & cleaned: 25 districts × 31 columns


,district,mean_hh_income,median_hh_income,avg_hh_size,gini_income,mean_per_capita_income,mean_hh_expenditure,median_hh_expenditure,gini_expenditure,mean_hh_per_capita_expenditure
0,Ampara,60474,42236,4.0,0.450,15856,52924,42587,0.36,13876
1,Anuradhapura,64409,46379,3.5,0.440,18473,52796,43362,0.35,15143
2,Badulla,66413,40063,3.6,0.530,18249,49971,34471,0.41,12907
3,Batticaloa,44686,35850,3.6,0.390,12561,41374,35409,0.31,11630
4,Colombo,132433,86981,3.8,0.470,34625,108893,81082,0.39,28470
5,Galle,70681,49719,3.6,0.400,19524,58504,45628,0.37,16161
6,Gampaha,100455,69729,3.7,0.400,27150,15618,84413,0.65,25205
7,Hambantota,68528,48621,3.8,0.440,18295,54169,42086,0.37,14461
8,Jaffna,55380,41822,4.0,0.440,13771,42213,36999,0.31,10497
9,Kalutara,84887,63586,4.0,0.410,21546,65970,54828,0.35,16744


## Section 3 — Feature Engineering & Normalisation

Derives 7 poverty indicators and scales them to `[0, 1]` using Min-Max normalisation.

| Feature | Logic |
|---|---|
| `poverty_proxy` | `1 / mean_per_capita_income` |
| `inequality_score` | `gini_income` |
| `expenditure_burden` | `mean_hh_expenditure / mean_hh_income` |
| `hh_size_pressure` | `avg_hh_size` |
| `price_pressure` | Latest month price index |
| `price_trend` | Linear slope over all monthly values |
| `price_volatility` | Std dev of monthly index |

In [5]:
scaler = MinMaxScaler()

df["price_volatility"]   = df[price_cols].std(axis=1)
df["poverty_proxy"]      = 1 / df["mean_per_capita_income"]
df["inequality_score"]   = df["gini_income"]
df["expenditure_burden"] = df["mean_hh_expenditure"] / df["mean_hh_income"]
df["hh_size_pressure"]   = df["avg_hh_size"]
df["price_pressure"]     = df[price_cols[-1]]

month_x = np.arange(len(price_cols))
df["price_trend"] = df[price_cols].apply(
    lambda row: np.polyfit(month_x, row.values.astype(float), 1)[0], axis=1)

feature_cols = ["poverty_proxy","inequality_score","expenditure_burden",
                "hh_size_pressure","price_pressure","price_trend","price_volatility"]

df_norm = df.copy()
df_norm[feature_cols] = scaler.fit_transform(df[feature_cols])

DEFAULT_RULES = {
    "poverty_proxy":0.30, "inequality_score":0.20, "expenditure_burden":0.20,
    "price_pressure":0.15, "price_trend":0.10, "hh_size_pressure":0.05,
}

def compute_risk_index(row, rules):
    return round(sum(row[f] * w for f, w in rules.items()), 4)

df_norm["base_risk_index"] = df_norm.apply(lambda r: compute_risk_index(r, DEFAULT_RULES), axis=1)

print(" Features engineered and normalised.")
df_norm[["district"] + feature_cols + ["base_risk_index"]].round(3)

 Features engineered and normalised.


,district,poverty_proxy,inequality_score,expenditure_burden,hh_size_pressure,price_pressure,price_trend,price_volatility,base_risk_index
0,Ampara,0.591,0.579,0.874,1.0,0.423,0.708,0.123,0.652
1,Anuradhapura,0.442,0.526,0.807,0.0,0.161,0.772,0.057,0.501
2,Badulla,0.453,1.000,0.725,0.2,0.419,0.710,0.121,0.625
3,Batticaloa,0.866,0.263,0.936,0.2,0.402,0.711,0.118,0.641
4,Colombo,0.021,0.684,0.810,0.6,1.000,0.559,0.267,0.541
5,Galle,0.394,0.316,0.817,0.2,0.486,0.677,0.138,0.495
6,Gampaha,0.154,0.316,0.000,0.4,0.955,0.573,0.256,0.330
7,Hambantota,0.451,0.526,0.771,0.6,0.126,0.783,0.049,0.522
8,Jaffna,0.750,0.526,0.737,1.0,0.161,0.773,0.057,0.629
9,Kalutara,0.313,0.368,0.755,1.0,0.764,0.617,0.208,0.545


## Section 4 — Enhanced Risk Score with Custom Rules

The enhanced risk score adds **penalty bonuses** on top of the base weighted score
to flag districts with compounding poverty indicators:

| Rule | Condition | Bonus |
|---|---|---|
| **Extreme Poverty** | Per capita income < Rs. 15,000 | +0.10 |
| **High Inequality** | Gini > 0.45 | +0.08 |
| **Expenditure Exceeds Income** | Expenditure/Income > 90% | +0.07 |
| **Large HH + Low Income** | HH size ≥ 4.0 AND income < Rs. 55,000 | +0.06 |
| **Rising Prices in Poor District** | Price trend rising + poverty score > 0.5 | +0.05 |

**Risk Tiers:**
- CRITICAL → score ≥ 0.70
- HIGH → score ≥ 0.55
- MODERATE → score ≥ 0.40
- LOW → score < 0.40

In [6]:
ALLOCATION_RULES = {
    "base_weights": {
        "poverty_proxy":0.30, "inequality_score":0.20, "expenditure_burden":0.20,
        "price_pressure":0.15, "price_trend":0.10, "hh_size_pressure":0.05,
    },
    "penalties": {
        "extreme_poverty_threshold":    15000,
        "extreme_poverty_bonus":        0.10,
        "high_gini_threshold":          0.45,
        "high_gini_bonus":              0.08,
        "expenditure_burden_threshold": 0.90,
        "expenditure_burden_bonus":     0.07,
        "large_hh_size_threshold":      4.0,
        "large_hh_income_threshold":    55000,
        "large_hh_bonus":               0.06,
        "rising_price_poverty_bonus":   0.05,
    },
    "equity_floor_pct": 0.015,   # 1.5% minimum allocation per district
}

def compute_enhanced_risk_score(row_norm, row_raw, rules):
    """Base weighted score + custom penalty bonuses."""
    p = rules["penalties"]

    # Layer 1: Base score
    base = sum(row_norm[f] * w for f, w in rules["base_weights"].items())

    # Layer 2: Penalty bonuses
    bonus = 0.0
    if row_raw["mean_per_capita_income"] < p["extreme_poverty_threshold"]:
        bonus += p["extreme_poverty_bonus"]
    if row_raw["gini_income"] > p["high_gini_threshold"]:
        bonus += p["high_gini_bonus"]
    if (row_raw["mean_hh_expenditure"] / row_raw["mean_hh_income"]) > p["expenditure_burden_threshold"]:
        bonus += p["expenditure_burden_bonus"]
    if (row_raw["avg_hh_size"] >= p["large_hh_size_threshold"] and
            row_raw["mean_hh_income"] < p["large_hh_income_threshold"]):
        bonus += p["large_hh_bonus"]
    if row_norm["price_trend"] > 0.5 and row_norm["poverty_proxy"] > 0.5:
        bonus += p["rising_price_poverty_bonus"]

    return round(min(base + bonus, 1.0), 4)

def classify_tier(score):
    if score >= 0.70: return "CRITICAL"
    if score >= 0.55: return "HIGH"
    if score >= 0.40: return "MODERATE"
    return "LOW"

df_alloc = df_norm.copy()
df_alloc["enhanced_risk_score"] = [
    compute_enhanced_risk_score(df_norm.loc[i], df.loc[i], ALLOCATION_RULES)
    for i in df_norm.index
]
df_alloc["risk_tier"] = df_alloc["enhanced_risk_score"].apply(classify_tier)

total_risk = df_alloc["enhanced_risk_score"].sum()
df_alloc["risk_pct"] = (df_alloc["enhanced_risk_score"] / total_risk * 100).round(3)

## Section 5 — Budget Allocation Engine

**How allocation works:**

1. **Equity Floor** — Every district receives a guaranteed minimum of **1.5%** of total budget
2. **Proportional Share** — Remaining budget distributed by each district's risk score share
3. **Final Allocation** = Floor Amount + Proportional Share

This ensures even low-risk districts receive some funding while
high-risk districts receive significantly more.

In [7]:
def allocate_budget(total_budget, df_risk, rules):
    """
    Allocate total_budget across districts based on enhanced risk scores.

    Parameters
    ----------
    total_budget : float   Total budget in Rs.
    df_risk      : df      Must contain 'district', 'enhanced_risk_score', 'risk_pct', 'risk_tier'
    rules        : dict    ALLOCATION_RULES

    Returns
    -------
    pd.DataFrame  Full allocation breakdown sorted by allocation (descending)
    """
    n            = len(df_risk)
    floor_pct    = rules["equity_floor_pct"]
    floor_amount = round(total_budget * floor_pct, 0)
    total_floor  = floor_amount * n
    remaining    = total_budget - total_floor

    out = df_risk[["district","enhanced_risk_score","risk_pct","risk_tier"]].copy()

    risk_sum = out["enhanced_risk_score"].sum()
    out["prop_share"]       = out["enhanced_risk_score"] / risk_sum
    out["prop_allocation"]  = (out["prop_share"] * remaining).round(0)
    out["floor_allocation"] = floor_amount
    out["total_allocation"] = out["floor_allocation"] + out["prop_allocation"]
    out["allocation_pct"]   = (out["total_allocation"] / total_budget * 100).round(3)

    # Per-household allocation
    out["alloc_per_hh"] = (
        out["total_allocation"].values / df[["avg_hh_size"]].values.flatten()
    ).round(0)

    # Fix rounding gap
    diff = total_budget - out["total_allocation"].sum()
    if abs(diff) > 0:
        top_idx = out["enhanced_risk_score"].idxmax()
        out.loc[top_idx, "total_allocation"] += diff

    return out.sort_values("total_allocation", ascending=False).reset_index(drop=True)

print("allocate_budget() function defined.")

allocate_budget() function defined.


##  Section 6 — Allocation Report & Visualisation

Generates:
- **Ranked allocation table** with Risk Score, Risk %, Amount (Rs.), Allocation %, Per-HH amount
- **4-panel chart**: allocation bar · risk vs allocation scatter · risk share pie · tier summary

In [8]:
def print_allocation_report(result, total_budget):
    """Print formatted allocation report."""
    print(f"\n{'═'*78}")
    print(f"   RESOURCE ALLOCATION REPORT")
    print(f"  Total Budget  : Rs. {total_budget:,.0f}")
    print(f"  No. Districts : {len(result)}")
    print(f"  Equity Floor  : Rs. {result['floor_allocation'].iloc[0]:,.0f} per district "
          f"({ALLOCATION_RULES['equity_floor_pct']*100:.1f}% each)")
    print(f"{'═'*78}")

    tbl = result[["district","risk_tier","enhanced_risk_score","risk_pct",
                  "total_allocation","allocation_pct","alloc_per_hh"]].copy()
    tbl.columns = ["District","Risk Tier","Risk Score","Risk %",
                   "Allocation (Rs.)","Alloc %","Per HH (Rs.)"]
    tbl["Allocation (Rs.)"] = tbl["Allocation (Rs.)"].apply(lambda x: f"{int(x):,}")
    tbl["Per HH (Rs.)"]     = tbl["Per HH (Rs.)"].apply(lambda x: f"{int(x):,}")
    tbl["Risk Score"]       = tbl["Risk Score"].round(4)
    tbl["Risk %"]           = tbl["Risk %"].round(2)
    tbl["Alloc %"]          = tbl["Alloc %"].round(2)
    tbl.index = range(1, len(tbl)+1)
    print(tbl.to_string())

    print(f"\n{'─'*78}")
    print(f"  Total Allocated : Rs. {result['total_allocation'].sum():,.0f}")
    for tier in [" CRITICAL"," HIGH"," MODERATE"," LOW"]:
        sub = result[result["risk_tier"] == tier]
        if not sub.empty:
            print(f"  {tier:22s} → {len(sub):2d} districts  |  "
                  f"Rs. {sub['total_allocation'].sum():>15,.0f}  "
                  f"({sub['allocation_pct'].sum():.1f}%)")
    print(f"{'─'*78}")

print("Report and visualisation functions defined.")

Report and visualisation functions defined.


##  Section 7 — Interactive Budget Input

Enter any budget amount and instantly receive:
- Full ranked allocation table (all 25 districts)
- Risk tier breakdown summary
- 4-panel visualisation

Type **`done`** to exit the loop.

**Example inputs:**
- `5000000000` → Rs. 5 Billion
- `2500000000` → Rs. 2.5 Billion
- `10000000000` → Rs. 10 Billion

In [9]:
while True:
    raw_input = input("\n Enter total budget in Rs. (e.g. 5000000000) or 'done' to stop: ").strip()

    if not raw_input or raw_input.lower() == "done":
        print(" Session ended.")
        break

    try:
        total_budget = float(raw_input.replace(",", ""))
    except ValueError:
        print("  Invalid input. Please enter a number like 5000000000.")
        continue

    allocation_result = allocate_budget(total_budget, df_alloc, ALLOCATION_RULES)
    print_allocation_report(allocation_result, total_budget)
    # visualise_allocation(allocation_result, total_budget)


 Enter total budget in Rs. (e.g. 5000000000) or 'done' to stop: done
 Session ended.


In [10]:
class PovertyRiskModel:

    def __init__(self, scaler, feature_cols, rules, encoder):
        self.scaler = scaler
        self.feature_cols = feature_cols
        self.rules = rules
        self.encoder = encoder


    def compute_risk(self, df_norm, df_raw):

        results = []

        p = self.rules["penalties"]
        weights = self.rules["base_weights"]

        for i in df_norm.index:

            row_norm = df_norm.loc[i]
            row_raw = df_raw.loc[i]

            # -------- Layer 1 : Base Score --------
            base_score = sum(row_norm[f] * w for f, w in weights.items())

            # -------- Layer 2 : Penalty Conditions --------
            bonus = 0

            if row_raw["mean_per_capita_income"] < p["extreme_poverty_threshold"]:
                bonus += p["extreme_poverty_bonus"]

            if row_raw["gini_income"] > p["high_gini_threshold"]:
                bonus += p["high_gini_bonus"]

            if (row_raw["mean_hh_expenditure"] / row_raw["mean_hh_income"]) > p["expenditure_burden_threshold"]:
                bonus += p["expenditure_burden_bonus"]

            if (row_raw["avg_hh_size"] >= p["large_hh_size_threshold"] and
                row_raw["mean_hh_income"] < p["large_hh_income_threshold"]):
                bonus += p["large_hh_bonus"]

            if row_norm["price_trend"] > 0.5 and row_norm["poverty_proxy"] > 0.5:
                bonus += p["rising_price_poverty_bonus"]

            score = min(base_score + bonus, 1.0)

            results.append(round(score,4))

        df_norm["enhanced_risk_score"] = results

        return df_norm

In [11]:
model = PovertyRiskModel(
    scaler=scaler,
    feature_cols=feature_cols,
    rules=ALLOCATION_RULES,
    encoder=encoder
)

with open("poverty_risk_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved as poverty_risk_model.pkl")

Model saved as poverty_risk_model.pkl


In [12]:


with open("poverty_risk_model.pkl", "rb") as f:
    model = pickle.load(f)

print("Model loaded successfully")

Model loaded successfully


In [13]:
risk_scores = model.compute_risk(df_norm, df)

print(risk_scores[:25])

        district  2024_jan  2024_feb  2024_mar  2024_apr  2024_may  2024_june  \
0         Ampara     17150     17110     16752     16607     16456      16599   
1   Anuradhapura     16604     16566     16219     16079     15933      16071   
2        Badulla     17140     17101     16742     16598     16447      16590   
3     Batticaloa     17107     17067     16709     16566     16415      16558   
4        Colombo     18350     18308     17924     17770     17608      17761   
5          Galle     17287     17247     16886     16740     16588      16732   
6        Gampaha     18256     18214     17832     17678     17517      17670   
7     Hambantota     16530     16493     16147     16008     15862      16000   
8         Jaffna     16604     16566     16219     16079     15933      16071   
9       Kalutara     17860     17819     17445     17295     17138      17287   
10         Kandy     17271     17231     16870     16725     16573      16717   
11       Kegalle     17790  